In [1]:
# Install required packages
!pip install ultralytics
!pip install opencv-python
!pip install pandas
!pip install numpy
!pip install matplotlib
!pip install torch torchvision
!pip install pillow

In [2]:
import pandas as pd
import os
import shutil
from pathlib import Path

def convert_to_yolo_format(csv_file, images_dir, output_dir):
    """
    Convert CSV annotation format to YOLO format
    CSV format: filename,width,height,class,xmin,ymin,xmax,ymax
    YOLO format: class_id center_x center_y width height (normalized)
    """
    # Read CSV file
    df = pd.read_csv(csv_file)
    
    # Create output directories
    labels_dir = os.path.join(output_dir, 'labels')
    images_output_dir = os.path.join(output_dir, 'images')
    os.makedirs(labels_dir, exist_ok=True)
    os.makedirs(images_output_dir, exist_ok=True)
    
    # Group by filename to handle multiple annotations per image
    grouped = df.groupby('filename')
    
    for filename, group in grouped:
        # Copy image to output directory
        src_image = os.path.join(images_dir, filename)
        dst_image = os.path.join(images_output_dir, filename)
        if os.path.exists(src_image):
            shutil.copy2(src_image, dst_image)
        
        # Create label file
        label_filename = os.path.splitext(filename)[0] + '.txt'
        label_path = os.path.join(labels_dir, label_filename)
        
        with open(label_path, 'w') as f:
            for _, row in group.iterrows():
                # Get image dimensions
                img_width = row['width']
                img_height = row['height']
                
                # Get bounding box coordinates
                xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']
                
                # Convert to YOLO format (normalized center coordinates and dimensions)
                center_x = (xmin + xmax) / 2.0 / img_width
                center_y = (ymin + ymax) / 2.0 / img_height
                width = (xmax - xmin) / img_width
                height = (ymax - ymin) / img_height
                
                # Class ID (0 for Graffiti)
                class_id = 0
                
                # Write to label file
                f.write(f"{class_id} {center_x:.6f} {center_y:.6f} {width:.6f} {height:.6f}\n")
    
    print(f"Conversion completed. Labels saved to {labels_dir}")
    return labels_dir, images_output_dir

# Convert training data
train_labels_dir, train_images_dir = convert_to_yolo_format(
    'Data/Bounding_boxes/train_labels.csv',
    'Data/images/train',
    'yolo_dataset/train'
)

# Convert test data
test_labels_dir, test_images_dir = convert_to_yolo_format(
    'Data/Bounding_boxes/test_labels.csv',
    'Data/images/test',
    'yolo_dataset/test'
)

Conversion completed. Labels saved to yolo_dataset/train\labels
Conversion completed. Labels saved to yolo_dataset/test\labels


In [3]:
# Create dataset.yaml file for YOLO training
dataset_config = """
path: ./yolo_dataset
train: train/images
val: test/images

nc: 1
names: ['Graffiti']
"""

with open('dataset.yaml', 'w') as f:
    f.write(dataset_config)

print("Dataset configuration file created: dataset.yaml")

Dataset configuration file created: dataset.yaml


In [4]:
import torch
import random
import numpy as np
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from sklearn.metrics import confusion_matrix
import seaborn as sns

In [5]:
def select_random_images(images_dir, labels_dir, num_images, output_dir):
    """
    Randomly select specified number of images and their corresponding labels
    """
    # Get all image files
    image_files = list(Path(images_dir).glob('*.jpg')) + list(Path(images_dir).glob('*.JPG'))
    
    # Randomly select images
    selected_images = random.sample(image_files, min(num_images, len(image_files)))
    
    # Create output directories
    selected_images_dir = os.path.join(output_dir, 'images')
    selected_labels_dir = os.path.join(output_dir, 'labels')
    os.makedirs(selected_images_dir, exist_ok=True)
    os.makedirs(selected_labels_dir, exist_ok=True)
    
    # Copy selected images and labels
    for img_path in selected_images:
        # Copy image
        shutil.copy2(img_path, os.path.join(selected_images_dir, img_path.name))
        
        # Copy corresponding label
        label_name = img_path.stem + '.txt'
        label_path = os.path.join(labels_dir, label_name)
        if os.path.exists(label_path):
            shutil.copy2(label_path, os.path.join(selected_labels_dir, label_name))
    
    return selected_images_dir, selected_labels_dir, [img.name for img in selected_images]

def train_yolo_model(train_images_dir, train_labels_dir, num_train_images, 
                     pretrained_model=None, epochs=50, iteration=1):
    """
    Train YOLO model with specified number of random training images
    """
    # Create iteration-specific directory
    iteration_dir = f'iteration_{iteration}'
    os.makedirs(iteration_dir, exist_ok=True)
    
    # Select random training images
    selected_train_dir = os.path.join(iteration_dir, 'train_selected')
    train_imgs_dir, train_lbls_dir, selected_images = select_random_images(
        train_images_dir, train_labels_dir, num_train_images, selected_train_dir
    )
    
    # Update dataset config for this iteration
    dataset_config_iter = f"""
path: ./{iteration_dir}
train: train_selected/images
val: ../yolo_dataset/test/images

nc: 1
names: ['Graffiti']
"""
    
    config_path = os.path.join(iteration_dir, 'dataset.yaml')
    with open(config_path, 'w') as f:
        f.write(dataset_config_iter)
    
    # Initialize model
    if pretrained_model and os.path.exists(pretrained_model):
        model = YOLO(pretrained_model)
        print(f"Loading pretrained model: {pretrained_model}")
    else:
        model = YOLO('yolov5s.pt')
        print("Using YOLOv5s pretrained weights")
    
    # Train model
    results = model.train(
        data=config_path,
        epochs=epochs,
        imgsz=640,
        batch=16,
        name=f'graffiti_detection_iter_{iteration}',
        project=iteration_dir,
        save=True,
        plots=True
    )
    
    # Save best model
    best_model_path = os.path.join(iteration_dir, f'best_iter_{iteration}.pt')
    shutil.copy2(results.save_dir / 'weights' / 'best.pt', best_model_path)
    
    return best_model_path, selected_images

# First iteration training
print("Starting Iteration 1...")
best_model_1, train_images_1 = train_yolo_model(
    train_images_dir, train_labels_dir, 400, epochs=10, iteration=1
)

Starting Iteration 1...
PRO TIP  Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.

Using YOLOv5s pretrained weights
Ultralytics 8.3.217  Python-3.9.23 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=iteration_1\dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=Fa

In [6]:
def calculate_iou(box1, box2):
    """
    Calculate Intersection over Union (IoU) between two bounding boxes
    box format: [x1, y1, x2, y2]
    """
    # Calculate intersection coordinates
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    # Calculate intersection area
    if x2 <= x1 or y2 <= y1:
        return 0.0
    
    intersection = (x2 - x1) * (y2 - y1)
    
    # Calculate union area
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union = area1 + area2 - intersection
    
    return intersection / union if union > 0 else 0.0

def yolo_to_xyxy(yolo_box, img_width, img_height):
    """
    Convert YOLO format (center_x, center_y, width, height) to (x1, y1, x2, y2)
    """
    center_x, center_y, width, height = yolo_box
    x1 = (center_x - width/2) * img_width
    y1 = (center_y - height/2) * img_height
    x2 = (center_x + width/2) * img_width
    y2 = (center_y + height/2) * img_height
    return [x1, y1, x2, y2]

In [7]:
def test_model_and_calculate_iou(model_path, test_images_dir, test_labels_dir, 
                                num_test_images, iteration):
    """
    Test model on random test images and calculate IoU for each detection
    """
    # Load model
    model = YOLO(model_path)
    
    # Select random test images
    test_selected_dir = f'iteration_{iteration}/test_selected'
    test_imgs_dir, test_lbls_dir, selected_test_images = select_random_images(
        test_images_dir, test_labels_dir, num_test_images, test_selected_dir
    )
    
    results_data = []
    detection_images = []
    
    for img_name in selected_test_images:
        img_path = os.path.join(test_imgs_dir, img_name)
        label_path = os.path.join(test_lbls_dir, img_name.replace('.jpg', '.txt').replace('.JPG', '.txt'))
        
        # Load image
        image = cv2.imread(img_path)
        img_height, img_width = image.shape[:2]
        
        # Run inference
        results = model(img_path, conf=0.25)
        
        # Load ground truth labels
        ground_truth_boxes = []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5:
                        _, center_x, center_y, width, height = map(float, parts)
                        gt_box = yolo_to_xyxy([center_x, center_y, width, height], img_width, img_height)
                        ground_truth_boxes.append(gt_box)
        
        # Process predictions
        max_iou = 0.0
        max_confidence = 0.0
        
        if len(results[0].boxes) > 0:
            for box in results[0].boxes:
                # Get prediction box coordinates
                pred_box = box.xyxy[0].cpu().numpy()
                confidence = box.conf[0].cpu().numpy()
                
                # Calculate IoU with all ground truth boxes
                for gt_box in ground_truth_boxes:
                    iou = calculate_iou(pred_box, gt_box)
                    if iou > max_iou:
                        max_iou = iou
                
                if confidence > max_confidence:
                    max_confidence = confidence
        
        # Store results
        results_data.append({
            'image_name': img_name,
            'confidence_value': max_confidence,
            'IoU_value': max_iou
        })
        
        # Save detection visualization for good samples
        if max_iou > 0.7:  # Save good detection examples
            annotated_img = results[0].plot()
            detection_images.append((img_name, annotated_img))
    
    # Create results DataFrame and save CSV
    results_df = pd.DataFrame(results_data)
    csv_path = f'iteration_{iteration}/test_results_iter_{iteration}.csv'
    results_df.to_csv(csv_path, index=False)
    
    # Save detection images
    detection_dir = f'iteration_{iteration}/detections'
    os.makedirs(detection_dir, exist_ok=True)
    for img_name, img_array in detection_images[:2]:  # Save top 2 good detections
        cv2.imwrite(os.path.join(detection_dir, f'detected_{img_name}'), img_array)
    
    # Calculate performance metrics
    iou_over_90 = (results_df['IoU_value'] > 0.9).sum()
    total_images = len(results_df)
    percentage_over_90 = (iou_over_90 / total_images) * 100
    
    print(f"Iteration {iteration} Results:")
    print(f"Images with IoU > 90%: {iou_over_90}/{total_images} ({percentage_over_90:.1f}%)")
    print(f"Average IoU: {results_df['IoU_value'].mean():.3f}")
    print(f"Average Confidence: {results_df['confidence_value'].mean():.3f}")
    
    return results_df, percentage_over_90

# Test first iteration
print("Testing Iteration 1...")
results_1, perf_1 = test_model_and_calculate_iou(
    best_model_1, test_images_dir, test_labels_dir, 40, 1
)

Testing Iteration 1...

image 1/1 D:\AI Train\AI4Engineer\iteration_1\test_selected\images\IMG_20180719_143723.jpg: 640x480 1 Graffiti, 50.8ms
Speed: 2.6ms preprocess, 50.8ms inference, 5.1ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 D:\AI Train\AI4Engineer\iteration_1\test_selected\images\IMG_20180721_090702.jpg: 640x480 1 Graffiti, 31.9ms
Speed: 2.1ms preprocess, 31.9ms inference, 3.2ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 D:\AI Train\AI4Engineer\iteration_1\test_selected\images\IMG_20100101_033637.jpg: 640x480 2 Graffitis, 28.6ms
Speed: 1.9ms preprocess, 28.6ms inference, 3.4ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 D:\AI Train\AI4Engineer\iteration_1\test_selected\images\IMG_20180717_120957.jpg: 640x480 2 Graffitis, 28.6ms
Speed: 1.8ms preprocess, 28.6ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 480)

image 1/1 D:\AI Train\AI4Engineer\iteration_1\test_selected\images\IMG_20180724_120949.jpg: 640x480 2 Graf

In [8]:
def iterative_training_loop(train_images_dir, train_labels_dir, test_images_dir, test_labels_dir,
                           target_percentage=80.0, max_iterations=10):
    """
    Iteratively train and test model until target performance is achieved
    """
    iteration = 1
    previous_model = None
    all_results = []
    
    while iteration <= max_iterations:
        print(f"\n{'='*50}")
        print(f"ITERATION {iteration}")
        print(f"{'='*50}")
        
        # Train model
        print(f"Training model for iteration {iteration}...")
        best_model, train_images = train_yolo_model(
            train_images_dir, train_labels_dir, 400, 
            pretrained_model=previous_model, 
            epochs=50, 
            iteration=iteration
        )
        
        # Test model
        print(f"Testing model for iteration {iteration}...")
        results_df, percentage_over_90 = test_model_and_calculate_iou(
            best_model, test_images_dir, test_labels_dir, 40, iteration
        )
        
        # Store results
        all_results.append({
            'iteration': iteration,
            'model_path': best_model,
            'percentage_over_90': percentage_over_90,
            'avg_iou': results_df['IoU_value'].mean(),
            'avg_confidence': results_df['confidence_value'].mean()
        })
        
        # Check if target achieved
        if percentage_over_90 >= target_percentage:
            print(f"\nTarget achieved! {percentage_over_90:.1f}% of images have IoU > 90%")
            break
        
        # Prepare for next iteration
        previous_model = best_model
        iteration += 1
        
        if iteration > max_iterations:
            print(f"\nMaximum iterations ({max_iterations}) reached.")
            break
    
    # Summary
    print(f"\n{'='*50}")
    print("TRAINING SUMMARY")
    print(f"{'='*50}")
    
    summary_df = pd.DataFrame(all_results)
    print(summary_df.to_string(index=False))
    
    # Save summary
    summary_df.to_csv('training_summary.csv', index=False)
    
    return all_results

# Run iterative training
training_results = iterative_training_loop(
    train_images_dir, train_labels_dir, 
    test_images_dir, test_labels_dir,
    target_percentage=80.0, 
    max_iterations=3
)


ITERATION 1
Training model for iteration 1...
PRO TIP  Replace 'model=yolov5s.pt' with new 'model=yolov5su.pt'.
YOLOv5 'u' models are trained with https://github.com/ultralytics/ultralytics and feature improved performance vs standard YOLOv5 models trained with https://github.com/ultralytics/yolov5.

Using YOLOv5s pretrained weights
Ultralytics 8.3.217  Python-3.9.23 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=iteration_1\dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=

In [17]:
def detect_graffiti_in_video(model_path, video_path, output_path, conf_threshold=0.25):
    """
    Detect graffiti in video using trained YOLO model
    """
    # Load model
    model = YOLO(model_path)
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    
    # Get video properties
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Setup video writer
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    frame_count = 0
    detection_count = 0
    
    print(f"Processing video: {video_path}")
    print(f"Output will be saved to: {output_path}")
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Run inference
        results = model(frame, conf=conf_threshold)
        
        # Draw detections
        annotated_frame = results[0].plot()
        
        # Count detections
        if len(results[0].boxes) > 0:
            detection_count += 1
        
        # Write frame
        out.write(annotated_frame)
        frame_count += 1
        
        if frame_count % 30 == 0:  # Print progress every 30 frames
            print(f"Processed {frame_count} frames, detections in {detection_count} frames")
    
    # Cleanup
    cap.release()
    out.release()
    
    print(f"Video processing completed!")
    print(f"Total frames: {frame_count}")
    print(f"Frames with detections: {detection_count}")
    print(f"Detection rate: {(detection_count/frame_count)*100:.1f}%")
    
    return output_path

# Process all available videos
video_files = [
    'videos/3413463-hd_1920_1080_30fps.mp4',
    'videos/4543511-hd_1080_1920_25fps.mp4', 
    'videos/854181-hd_1920_1080_25fps.mp4',
    'videos/9724130-hd_1440_1080_30fps.mp4'
]

# Get the best model from training results
best_iteration = max(training_results, key=lambda x: x['percentage_over_90'])
final_model_path = best_iteration['model_path']

print(f"Using best model from iteration {best_iteration['iteration']}: {final_model_path}")

# Process each video
video_results = []
for i, video_path in enumerate(video_files, 1):
    if os.path.exists(video_path):
        output_path = f'video_detection_output_{i}.mp4'
        result_path = detect_graffiti_in_video(final_model_path, video_path, output_path)
        video_results.append(result_path)
        print(f"Completed processing video {i}/4\n")
    else:
        print(f"Video not found: {video_path}")

Using best model from iteration 1: iteration_1\best_iter_1.pt
Processing video: videos/3413463-hd_1920_1080_30fps.mp4
Output will be saved to: video_detection_output_1.mp4

0: 384x640 5 Graffitis, 174.9ms
Speed: 28.8ms preprocess, 174.9ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 Graffitis, 73.9ms
Speed: 3.6ms preprocess, 73.9ms inference, 5.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 Graffitis, 33.4ms
Speed: 2.9ms preprocess, 33.4ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Graffitis, 33.3ms
Speed: 2.5ms preprocess, 33.3ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Graffitis, 33.3ms
Speed: 2.5ms preprocess, 33.3ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Graffitis, 31.8ms
Speed: 2.4ms preprocess, 31.8ms inference, 1.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 Graffitis, 31.8ms
Speed: 2.6ms prepro

In [18]:
def create_performance_plots():
    """
    Create visualization plots for training performance
    """
    # Load training summary
    summary_df = pd.read_csv('training_summary.csv')
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Plot 1: IoU over iterations
    axes[0,0].plot(summary_df['iteration'], summary_df['avg_iou'], 'b-o', linewidth=2, markersize=8)
    axes[0,0].set_title('Average IoU per Iteration', fontsize=14, fontweight='bold')
    axes[0,0].set_xlabel('Iteration')
    axes[0,0].set_ylabel('Average IoU')
    axes[0,0].grid(True, alpha=0.3)
    axes[0,0].set_ylim(0, 1)
    
    # Plot 2: Percentage over 90% IoU
    axes[0,1].bar(summary_df['iteration'], summary_df['percentage_over_90'], 
                  color='green', alpha=0.7, edgecolor='black')
    axes[0,1].axhline(y=80, color='red', linestyle='--', linewidth=2, label='Target (80%)')
    axes[0,1].set_title('Percentage of Images with IoU > 90%', fontsize=14, fontweight='bold')
    axes[0,1].set_xlabel('Iteration')
    axes[0,1].set_ylabel('Percentage (%)')
    axes[0,1].legend()
    axes[0,1].grid(True, alpha=0.3)
    
    # Plot 3: Average Confidence
    axes[1,0].plot(summary_df['iteration'], summary_df['avg_confidence'], 'r-s', linewidth=2, markersize=8)
    axes[1,0].set_title('Average Confidence per Iteration', fontsize=14, fontweight='bold')
    axes[1,0].set_xlabel('Iteration')
    axes[1,0].set_ylabel('Average Confidence')
    axes[1,0].grid(True, alpha=0.3)
    axes[1,0].set_ylim(0, 1)
    
    # Plot 4: Combined metrics
    ax2 = axes[1,1].twinx()
    line1 = axes[1,1].plot(summary_df['iteration'], summary_df['avg_iou'], 'b-o', label='Avg IoU')
    line2 = ax2.plot(summary_df['iteration'], summary_df['percentage_over_90'], 'g-s', label='% > 90% IoU')
    
    axes[1,1].set_xlabel('Iteration')
    axes[1,1].set_ylabel('Average IoU', color='blue')
    ax2.set_ylabel('Percentage (%)', color='green')
    axes[1,1].set_title('Combined Performance Metrics', fontsize=14, fontweight='bold')
    
    # Combine legends
    lines1, labels1 = axes[1,1].get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    axes[1,1].legend(lines1 + lines2, labels1 + labels2, loc='center right')
    
    plt.tight_layout()
    plt.savefig('training_performance_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

# Create performance plots
create_performance_plots()

<Figure size 1500x1000 with 5 Axes>